# Exploratory Data Analysis — Teen Mental Health Dataset

This notebook performs a comprehensive exploratory data analysis (EDA) on the Teen Mental Health dataset.  
The ultimate goal of this project is to build a **Logistic Regression** model to predict the `depression_label` target variable.

---

**Dataset:** `data/Teen_Mental_Health_Dataset.csv`  
**Target variable:** `depression_label` — binary (0 = no depression, 1 = depression)  
**Records:** ~1,200 teenagers  

### Notebook outline
1. Library imports  
2. Data loading  
3. Initial inspection  
4. Missing value analysis  
5. Target variable distribution  
6. Univariate analysis — numerical features  
7. Univariate analysis — categorical features  
8. Bivariate analysis — numerical vs. target  
9. Bivariate analysis — categorical vs. target  
10. Correlation matrix  
11. Scatter matrix — top correlated features  
12. Summary and key insights  


---
## 1. Library Imports

In [ ]:
# Standard library
import os
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization — %matplotlib inline enables plots to render inside the notebook
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pandas.plotting import scatter_matrix
import seaborn as sns

# Notebook display utilities
from IPython.display import display

# -- Visual theme ------------------------------------------------------------
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 100,
})

# -- Pandas display options --------------------------------------------------
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# -- Output directory for saved figures --------------------------------------
FIGURES_DIR = os.path.join('..', 'outputs', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

print('Libraries imported successfully.')

---
## 2. Data Loading

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'Teen_Mental_Health_Dataset.csv')

df = pd.read_csv(DATA_PATH)

print(f'Dataset shape : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Columns       : {list(df.columns)}')

---
## 3. Initial Inspection

In [ ]:
# Preview the first rows to understand data structure and value formats
print('First 5 rows:')
display(df.head())

In [ ]:
# Data types and non-null counts per column
print('Data types and non-null counts:')
df.info()

In [ ]:
# Descriptive statistics for numerical columns
# Helps detect outliers, scale differences, and skewness at a glance
print('Descriptive statistics (numerical features):')
display(df.describe())

In [ ]:
# Frequency summary for categorical columns
print('Descriptive statistics (categorical features):')
display(df.describe(include='object'))

---
## 4. Missing Value Analysis

In [ ]:
# Count and percentage of null values per column
missing_counts = df.isnull().sum()
missing_pct    = (missing_counts / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing (%)' : missing_pct,
}).query('`Missing Count` > 0')

if missing_report.empty:
    print('No missing values found -- dataset is complete.')
else:
    print(f'{len(missing_report)} column(s) contain missing values:')
    display(missing_report)

---
## 5. Target Variable Distribution

Before any modelling, it is critical to understand whether the classes are **balanced or imbalanced**,  
as this directly affects the choice of evaluation metrics and resampling strategy.

In [ ]:
target_counts = df['depression_label'].value_counts().sort_index()
target_pct    = df['depression_label'].value_counts(normalize=True).sort_index() * 100

CLASS_LABELS = {0: 'No Depression (0)', 1: 'Depression (1)'}
PALETTE      = sns.color_palette('Set2', 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# -- Bar chart ---------------------------------------------------------------
bars = axes[0].bar(
    [CLASS_LABELS[k] for k in target_counts.index],
    target_counts.values,
    color=PALETTE,
)
for bar, count in zip(bars, target_counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        str(count), ha='center', va='bottom', fontweight='bold',
    )
axes[0].set_title('Class Count')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, target_counts.max() * 1.15)

# -- Pie chart ---------------------------------------------------------------
axes[1].pie(
    target_pct.values,
    labels=[CLASS_LABELS[k] for k in target_pct.index],
    autopct='%1.1f%%',
    colors=PALETTE,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
)
axes[1].set_title('Class Proportion')

fig.suptitle('Target Variable Distribution -- depression_label',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '01_target_distribution.png'), bbox_inches='tight')
plt.show()

for label in [0, 1]:
    print(f'  Class {label} ({CLASS_LABELS[label]}): {target_counts[label]:>4} ({target_pct[label]:.1f}%)')

---
## 6. Univariate Analysis -- Numerical Features

Histograms help us assess the **shape** (normal, skewed, bimodal) and **range** of each numerical variable.  
The red dashed line marks the mean.

In [ ]:
# Identify numerical columns, excluding the binary target
num_cols = df.select_dtypes(include='number').columns.drop('depression_label').tolist()
print(f'Numerical features ({len(num_cols)}): {num_cols}')

N_COLS = 3
N_ROWS = -(-len(num_cols) // N_COLS)  # ceiling division

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(15, N_ROWS * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(
        df[col].dropna(), bins=25,
        color=sns.color_palette('Set2')[i % 8],
        edgecolor='white',
    )
    mean_val = df[col].mean()
    axes[i].axvline(mean_val, color='crimson', linestyle='--', linewidth=1.5,
                    label=f'Mean: {mean_val:.2f}')
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=9)

# Hide unused subplot slots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Numerical Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '02_numeric_distributions.png'), bbox_inches='tight')
plt.show()

---
## 7. Univariate Analysis -- Categorical Features

Count plots show how records are distributed across the categories of each categorical variable.

In [ ]:
# Identify categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')

fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 5))
if len(cat_cols) == 1:
    axes = [axes]

for i, col in enumerate(cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=axes[i],
                  palette='Set2', hue=col, legend=False)
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=30)

    # Annotate bar heights
    for bar in axes[i].patches:
        h = bar.get_height()
        if h > 0:
            axes[i].text(
                bar.get_x() + bar.get_width() / 2, h + 2,
                str(int(h)), ha='center', va='bottom', fontsize=9,
            )

fig.suptitle('Categorical Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '03_categorical_distributions.png'), bbox_inches='tight')
plt.show()

---
## 8. Bivariate Analysis -- Numerical Features vs. Target

Box plots compare the **distribution of each numerical feature across the two classes**.  
Features where the medians differ significantly between classes are likely strong predictors.

In [ ]:
# Color map: green = no depression, orange = depression
BP_PALETTE = {0: sns.color_palette('Set2')[0], 1: sns.color_palette('Set2')[1]}

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(15, N_ROWS * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(
        data=df, x='depression_label', y=col,
        palette=BP_PALETTE, hue='depression_label',
        legend=False, ax=axes[i],
    )
    axes[i].set_title(col)
    axes[i].set_xlabel('depression_label  (0 = No  |  1 = Yes)')
    axes[i].set_ylabel(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Numerical Features by Depression Label (Box Plots)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '04_bivariate_boxplots.png'), bbox_inches='tight')
plt.show()

---
## 9. Bivariate Analysis -- Categorical Features vs. Target

Grouped bar charts show the **depression rate within each category**.  
A category where the depression proportion differs noticeably signals a relevant feature.

In [ ]:
fig, axes = plt.subplots(1, len(cat_cols), figsize=(6 * len(cat_cols), 5))
if len(cat_cols) == 1:
    axes = [axes]

for i, col in enumerate(cat_cols):
    # Normalise within each category (row-wise) to get proportions
    cross = pd.crosstab(df[col], df['depression_label'], normalize='index') * 100
    cross.plot(
        kind='bar', ax=axes[i],
        colormap='Set2', edgecolor='white', width=0.65,
    )
    axes[i].set_title(f'Depression Rate by {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('% within group')
    axes[i].legend(['No Depression (0)', 'Depression (1)'], fontsize=9)
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].yaxis.set_major_formatter(mticker.PercentFormatter())

fig.suptitle('Depression Rate by Categorical Feature', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '05_categorical_vs_target.png'), bbox_inches='tight')
plt.show()

---
## 10. Correlation Matrix

The heatmap shows **pairwise Pearson correlations** among all numerical variables including the target.  
Key observations:
- Features with high absolute correlation to `depression_label` are strong candidates as predictors.
- Feature pairs with very high mutual correlation may cause multicollinearity in logistic regression.

In [ ]:
corr_matrix = df[num_cols + ['depression_label']].corr()

# Use upper triangle as mask to avoid redundant information
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(11, 8))
sns.heatmap(
    corr_matrix, mask=mask,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8},
)
plt.title('Correlation Matrix -- Numerical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '06_correlation_matrix.png'), bbox_inches='tight')
plt.show()

# Rank features by absolute correlation with the target
target_corr = (
    corr_matrix['depression_label']
    .drop('depression_label')
    .sort_values(key=abs, ascending=False)
)

print('\nFeature correlations with depression_label (sorted by |r|):')
print('-' * 50)
for feat, val in target_corr.items():
    direction = 'up' if val > 0 else 'dn'
    bar = '#' * int(abs(val) * 20)
    print(f'  [{direction}] {feat:<30} {val:+.3f}  {bar}')

---
## 11. Scatter Matrix -- Top Correlated Features

A scatter matrix of the **4 most correlated features** gives a visual feel for linear separability  
and interactions between the strongest predictors.  
Green = No Depression  |  Orange = Depression

In [ ]:
# Select the top-4 features by absolute correlation with the target
top_features = target_corr.abs().nlargest(4).index.tolist()
print(f'Top-4 features: {top_features}')

# Map class labels to colours for the scatter matrix
point_colors = df['depression_label'].map({0: '#66C2A5', 1: '#FC8D62'})

scatter_matrix(
    df[top_features],
    c=point_colors,
    alpha=0.4,
    figsize=(12, 10),
    diagonal='hist',
    s=15,
)

plt.suptitle(
    'Scatter Matrix -- Top 4 Features  (Green = No Depression | Orange = Depression)',
    y=1.01, fontsize=13, fontweight='bold',
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '07_scatter_matrix.png'), bbox_inches='tight')
plt.show()

---
## 12. Summary and Key Insights

In [ ]:
print('=' * 60)
print('              EDA SUMMARY')
print('=' * 60)
print(f'  Total records          : {len(df):>6,}')
print(f'  Total features         : {df.shape[1] - 1:>6}')
print(f'  Numerical features     : {len(num_cols):>6}')
print(f'  Categorical features   : {len(cat_cols):>6}')
print(f'  Missing values         : {df.isnull().sum().sum():>6}')
print()
print('  Target class distribution:')
for label, count in df['depression_label'].value_counts().sort_index().items():
    pct  = count / len(df) * 100
    bar  = '#' * int(pct / 3)
    name = CLASS_LABELS[label]
    print(f'    {name:<22}: {count:>4} ({pct:5.1f}%)  {bar}')
print()
print('  Feature correlations with depression_label:')
for feat, val in target_corr.items():
    direction = 'positive' if val > 0 else 'negative'
    print(f'    {feat:<30} {val:+.3f}  ({direction})')
print('=' * 60)
print()
print('Next steps:')
print('  1. Encode categorical features (One-Hot / Ordinal Encoding)')
print('  2. Scale numerical features (StandardScaler / MinMaxScaler)')
print('  3. Stratified train / validation / test split')
print('  4. Train Logistic Regression and evaluate (ROC-AUC, F1, Confusion Matrix)')